<a href="https://colab.research.google.com/github/https-deeplearning-ai/tensorflow-1-public/blob/master/C1/W2/ungraded_labs/C1_W2_Lab_1_beyond_hello_world.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Start Coding

Let's start with our import of TensorFlow.

In [ ]:
import tensorflow as tf

print(tf.__version__)

The [Fashion MNIST dataset](https://github.com/zalandoresearch/fashion-mnist) is a collection of grayscale 28x28 pixel clothing images. Each image is associated with a label as shown in this table⁉

| Label | Description |
| --- | --- |
| 0 | T-shirt/top |
| 1 | Trouser |
| 2 | Pullover |
| 3 | Dress |
| 4 | Coat |
| 5 | Sandal |
| 6 | Shirt |
| 7 | Sneaker |
| 8 | Bag |
| 9 | Ankle boot |

This dataset is available directly in the [tf.keras.datasets](https://www.tensorflow.org/api_docs/python/tf/keras/datasets) API and you load it like this:

In [ ]:
# Load the Fashion MNIST dataset
fmnist = tf.keras.datasets.fashion_mnist

Calling `load_data()` on this object will give you two tuples with two lists each. These will be the training and testing values for the graphics that contain the clothing items and their labels.


In [ ]:
# Load the training and test split of the Fashion MNIST dataset
(training_images, training_labels), (test_images, test_labels) = fmnist.load_data()

What do these values look like? Let's print a training image (both as an image and a numpy array), and a training label to see. Experiment with different indices in the array. For example, also take a look at index `42`. That's a different boot than the one at index `0`.


In [ ]:
from matplotlib.colors import Colormap
import numpy as np
import matplotlib.pyplot as plt

# You can put between 0 to 59999 here
index = 100

# Set number of characters per row when printing
np.set_printoptions(linewidth=320)

# Print the label and image
print(f'LABEL: {training_labels[index]}')
print(f'\nIMAGE PIXEL ARRAY:\n {training_images[index]}')

# Visualize the image
plt.figure(figsize=(0.5, 0.3))
plt.imshow(training_images[index])

You'll notice that all of the values in the number are between 0 and 255. If you are training a neural network especially in image processing, for various reasons it will usually learn better if you scale all values to between 0 and 1. It's a process called _normalization_ and fortunately in Python, it's easy to normalize an array without looping. You do it like this:

In [ ]:
# Normalize the pixel values of the train and test images
training_images  = training_images / 255.0
test_images = test_images / 255.0

Now you might be wondering why the dataset is split into two: training and testing? Remember we spoke about this in the intro? The idea is to have 1 set of data for training, and then another set of data that the model hasn't yet seen. This will be used to evaluate how good it would be at classifying values.

Let's now design the model. There's quite a few new concepts here. But don't worry, you'll get the hang of them.

In [ ]:
training_images[1]

In [ ]:
# Build the classification model
model = tf.keras.Sequential([tf.keras.layers.Flatten(),
                                    tf.keras.layers.Dense(128, activation=tf.nn.relu),
                                    tf.keras.layers.Dense(10, activation=tf.nn.softmax)])

In [ ]:
# Build the classification model
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.models import Sequential

model = Sequential()
model.add(Flatten(input_shape=(28, 28)))
model.add(Dense(128, activation='relu'))
model.add(Dense(256, activation='relu'))
model.add(Dense(10, activation='softmax'))

[Sequential](https://keras.io/api/models/sequential/): That defines a sequence of layers in the neural network.

[Flatten](https://keras.io/api/layers/reshaping_layers/flatten/): Remember earlier where our images were a 28x28 pixel matrix when you printed them out? Flatten just takes that square and turns it into a 1-dimensional array.

[Dense](https://keras.io/api/layers/core_layers/dense/): Adds a layer of neurons

Each layer of neurons need an [activation function](https://keras.io/api/layers/activations/) to tell them what to do. There are a lot of options, but just use these for now:

[ReLU](https://keras.io/api/layers/activations/#relu-function) effectively means:

```
if x > 0:
  return x

else:
  return 0
```

In other words, it only passes values greater than 0 to the next layer in the network.

[Softmax](https://keras.io/api/layers/activations/#softmax-function) takes a list of values and scales these so the sum of all elements will be equal to 1. When applied to model outputs, you can think of the scaled values as the probability for that class. For example, in your classification model which has 10 units in the output dense layer, having the highest value at `index = 4` means that the model is most confident that the input clothing image is a coat. If it is at index = 5, then it is a sandal, and so forth. See the short code block below which demonstrates these concepts. You can also watch this [lecture](https://www.youtube.com/watch?v=LLux1SW--oM&ab_channel=DeepLearningAI) if you want to know more about the Softmax function and how the values are computed.


In [ ]:
# Declare sample inputs and convert to a tensor
inputs = np.array([[1.0, 3.0, 4.0, 2.0]])
inputs = tf.convert_to_tensor(inputs)
print(f'input to softmax function: {inputs.numpy()}')

# Feed the inputs to a softmax activation function
outputs = tf.keras.activations.softmax(inputs)
print(f'output of softmax function: {outputs.numpy()}')

# Get the sum of all values after the softmax
sum = tf.reduce_sum(outputs)
print(f'sum of outputs: {sum}')

# Get the index with highest value
prediction = np.argmax(outputs)
print(f'class with highest probability: {prediction}')

The next thing to do, now that the model is defined, is to actually build it. You do this by compiling it with an optimizer and loss function as before -- and then you train it by calling `model.fit()` asking it to fit your training data to your training labels. It will figure out the relationship between the training data and its actual labels so in the future if you have inputs that looks like the training data, then it can predict what the label for that input is.

In [ ]:
from tensorflow.keras.layers import Flatten, Dense, Input
from tensorflow.keras.models import Sequential

# Build the classification model (re-defined to ensure 'model' is always available)
model = Sequential([
    Input(shape=(28, 28)), # Specify input shape using Input layer as recommended
    Flatten(),
    Dense(128, activation='relu'),
    Dense(256, activation='relu'),
    Dense(10, activation='softmax')
])

model.compile(optimizer = tf.optimizers.Adam(),
              loss = 'sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.fit(training_images, training_labels, epochs=30, validation_split=0.2)

Once it's done training -- you should see an accuracy value at the end of the final epoch. It might look something like `0.9098`. This tells you that your neural network is about 91% accurate in classifying the training data. That is, it figured out a pattern match between the image and the labels that worked 91% of the time. Not great, but not bad considering it was only trained for 5 epochs and done quite quickly.

But how would it work with unseen data? That's why we have the test images and labels. We can call [`model.evaluate()`](https://keras.io/api/models/model_training_apis/#evaluate-method) with this test dataset as inputs and it will report back the loss and accuracy of the model. Let's give it a try:

In [ ]:
# Evaluate the model on unseen data
model.evaluate(test_images, test_labels)

You can expect the accuracy here to be about `0.88` which means it was 88% accurate on the entire test set. As expected, it probably would not do as well with *unseen* data as it did with data it was trained on!  As you go through this course, you'll look at ways to improve this.

# Exploration Exercises

To explore further and deepen your understanding, try the below exercises:

### Exercise 1:
For this first exercise run the below code: It creates a set of classifications for each of the test images, and then prints the first entry in the classifications. The output, after you run it is a list of numbers. Why do you think this is, and what do those numbers represent?

In [ ]:
classifications = model.predict(test_images)

print(classifications[0])

**Hint:** try running `print(test_labels[0])` -- and you'll get a `9`. Does that help you understand why this list looks the way it does?

In [ ]:
print(test_labels[0])

### E1Q1: What does this list represent?


1.   It's 10 random meaningless values
2.   It's the first 10 classifications that the computer made
3.   It's the probability that this item is each of the 10 classes


#### Answer:
The correct answer is (3)

The output of the model is a list of 10 numbers. These numbers are a probability that the value being classified is the corresponding value (https://github.com/zalandoresearch/fashion-mnist#labels), i.e. the first value in the list is the probability that the image is of a '0' (T-shirt/top), the next is a '1' (Trouser) etc. Notice that they are all VERY LOW probabilities.

For index 9 (Ankle boot), the probability was in the 90's, i.e. the neural network is telling us that the image is most likely an ankle boot.

<details><summary>Click for Answer</summary>
<p>

#### Answer:
The correct answer is (3)

The output of the model is a list of 10 numbers. These numbers are a probability that the value being classified is the corresponding value (https://github.com/zalandoresearch/fashion-mnist#labels), i.e. the first value in the list is the probability that the image is of a '0' (T-shirt/top), the next is a '1' (Trouser) etc. Notice that they are all VERY LOW probabilities.

For index 9 (Ankle boot), the probability was in the 90's, i.e. the neural network is telling us that the image is most likely an ankle boot.

</p>
</details>

### E1Q2: How do you know that this list tells you that the item is an ankle boot?


1.   There's not enough information to answer that question
2.   The 10th element on the list is the biggest, and the ankle boot is labelled 9
2.   The ankle boot is label 9, and there are 0->9 elements in the list


#### Answer
The correct answer is (2). Both the list and the labels are 0 based, so the ankle boot having label 9 means that it is the 10th of the 10 classes. The list having the 10th element being the highest value means that the Neural Network has predicted that the item it is classifying is most likely an ankle boot

<details><summary>Click for Answer</summary>
<p>

#### Answer
The correct answer is (2). Both the list and the labels are 0 based, so the ankle boot having label 9 means that it is the 10th of the 10 classes. The list having the 10th element being the highest value means that the Neural Network has predicted that the item it is classifying is most likely an ankle boot

</p>
</details>

### Exercise 2:
Let's now look at the layers in your model. Experiment with different values for the dense layer with 512 neurons. What different results do you get for loss, training time etc? Why do you think that's the case?


In [ ]:
fmnist = tf.keras.datasets.fashion_mnist

(training_images, training_labels) ,  (test_images, test_labels) = fmnist.load_data()

training_images = training_images/255.0
test_images = test_images/255.0

model = tf.keras.models.Sequential([tf.keras.layers.Flatten(),
                                    tf.keras.layers.Dense(512, activation=tf.nn.relu), # Try experimenting with this layer
                                    tf.keras.layers.Dense(10, activation=tf.nn.softmax)])

model.compile(optimizer = 'adam',
              loss = 'sparse_categorical_crossentropy')

model.fit(training_images, training_labels, epochs=5)

model.evaluate(test_images, test_labels)

classifications = model.predict(test_images)

print(classifications[0])
print(test_labels[0])

### E2Q1: Increase to 1024 Neurons -- What's the impact?

1. Training takes longer, but is more accurate
2. Training takes longer, but no impact on accuracy
3. Training takes the same time, but is more accurate


<details><summary>Click for Answer</summary>
<p>

#### Answer
The correct answer is (1) by adding more Neurons we have to do more calculations, slowing down the process, but in this case they have a good impact -- we do get more accurate. That doesn't mean it's always a case of 'more is better', you can hit the law of diminishing returns very quickly!

</p>
</details>

### Exercise 3:

### E3Q1: What would happen if you remove the Flatten() layer. Why do you think that's the case?

<details><summary>Click for Answer</summary>
<p>

#### Answer
You get an error about the shape of the data. It may seem vague right now, but it reinforces the rule of thumb that the first layer in your network should be the same shape as your data. Right now our data is 28x28 images, and 28 layers of 28 neurons would be infeasible, so it makes more sense to 'flatten' that 28,28 into a 784x1. Instead of writing all the code to handle that ourselves, we add the Flatten() layer at the begining, and when the arrays are loaded into the model later, they'll automatically be flattened for us.

</p>
</details>

In [ ]:
fmnist = tf.keras.datasets.fashion_mnist

(training_images, training_labels) ,  (test_images, test_labels) = fmnist.load_data()

training_images = training_images/255.0
test_images = test_images/255.0

model = tf.keras.models.Sequential([tf.keras.layers.Flatten(), # This layer has been re-enabled to fix the error
                                    tf.keras.layers.Dense(64, activation=tf.nn.relu),
                                    tf.keras.layers.Dense(10, activation=tf.nn.softmax)])

model.compile(optimizer = 'adam',
              loss = 'sparse_categorical_crossentropy')

model.fit(training_images, training_labels, epochs=5)

model.evaluate(test_images, test_labels)

classifications = model.predict(test_images)

print(classifications[0])
print(test_labels[0])

### Exercise 3: E3Q1 - What would happen if you remove the Flatten() layer. Why do you think that's the case?

**Answer:** As you can see from the execution results of the cell for Exercise 3, removing the `Flatten()` layer results in a `ValueError`. Specifically, `ValueError: Argument 'output' must have rank (ndim) 'target.ndim - 1'. Received: target.shape=(32,), output.shape=(32, 28, 10)`. This error occurs because your input data (images) are 2D arrays (28x28 pixels), but the subsequent `Dense` layers expect 1D input. The `Flatten()` layer's job is to transform that 2D image into a 1D array (784 features), making it compatible with the `Dense` layers. Without it, the dimensions don't match, causing the model to fail.

### Exercise 3: E3Q1 - What would happen if you remove the Flatten() layer. Why do you think that's the case?

**Answer:** As you can see from the execution results of the cell for Exercise 3, removing the `Flatten()` layer results in a `ValueError`. Specifically, `ValueError: Argument 'output' must have rank (ndim) 'target.ndim - 1'. Received: target.shape=(32,), output.shape=(32, 28, 10)`. This error occurs because your input data (images) are 2D arrays (28x28 pixels), but the subsequent `Dense` layers expect 1D input. The `Flatten()` layer's job is to transform that 2D image into a 1D array (784 features), making it compatible with the `Dense` layers. Without it, the dimensions don't match, causing the model to fail.

### Exercise 4:

Consider the final (output) layers. Why are there 10 of them? What would happen if you had a different amount than 10? For example, try training the network with 5.

<details><summary>Click for Answer</summary>
<p>

#### Answer
You get an error as soon as it finds an unexpected value. Another rule of thumb -- the number of neurons in the last layer should match the number of classes you are classifying for. In this case it's the digits 0-9, so there are 10 of them, hence you should have 10 neurons in your final layer.

</p>
</details>

In [ ]:
fmnist = tf.keras.datasets.fashion_mnist

(training_images, training_labels) ,  (test_images, test_labels) = fmnist.load_data()

training_images = training_images/255.0
test_images = test_images/255.0

model = tf.keras.models.Sequential([tf.keras.layers.Flatten(),
                                    tf.keras.layers.Dense(64, activation=tf.nn.relu),
                                    tf.keras.layers.Dense(10, activation=tf.nn.softmax) # Changed back to 10 neurons to fix the error
                                  ])

model.compile(optimizer = 'adam',
              loss = 'sparse_categorical_crossentropy')

model.fit(training_images, training_labels, epochs=5)

model.evaluate(test_images, test_labels)

classifications = model.predict(test_images)

print(classifications[0])
print(test_labels[0])

### Exercise 4: E4Q1 - What would happen if you had a different amount than 10 neurons in the final layer? For example, try training the network with 5.

**Answer:** When you train the network with 5 neurons in the final layer, as seen in the execution output for Exercise 4, the training loss quickly becomes `nan` (Not a Number), and the classifications also become `[nan nan nan nan nan]`. This indicates that the model is unable to learn effectively. The reason for this is that the number of neurons in the final layer of a classification network must match the number of classes you are trying to predict. Since the Fashion MNIST dataset has 10 distinct classes (0-9), having only 5 output neurons means the model cannot represent all possible output categories, leading to this failure.

### Exercise 4: E4Q1 - What would happen if you had a different amount than 10 neurons in the final layer? For example, try training the network with 5.

**Answer:** When you train the network with 5 neurons in the final layer, as seen in the execution output for Exercise 4, the training loss quickly becomes `nan` (Not a Number), and the classifications also become `[nan nan nan nan nan]`. This indicates that the model is unable to learn effectively. The reason for this is that the number of neurons in the final layer of a classification network must match the number of classes you are trying to predict. Since the Fashion MNIST dataset has 10 distinct classes (0-9), having only 5 output neurons means the model cannot represent all possible output categories, leading to this failure.

### Exercise 5:

Consider the effects of additional layers in the network. What will happen if you add another layer between the one with 512 and the final layer with 10.

<details><summary>Click for Answer</summary>
<p>

#### Answer
There isn't a significant impact -- because this is relatively simple data. For far more complex data (including color images to be classified as flowers that you'll see in the next lesson), extra layers are often necessary.

</p>
</details>

In [ ]:
fmnist = tf.keras.datasets.fashion_mnist

(training_images, training_labels) ,  (test_images, test_labels) = fmnist.load_data()

training_images = training_images/255.0
test_images = test_images/255.0

model = tf.keras.models.Sequential([tf.keras.layers.Flatten(),
                                    tf.keras.layers.Dense(512, activation=tf.nn.relu), # Original layer
                                    tf.keras.layers.Dense(128, activation=tf.nn.relu), # Added for Exercise 5
                                    tf.keras.layers.Dense(10, activation=tf.nn.softmax) # Final layer
                                  ])

model.compile(optimizer = 'adam',
              loss = 'sparse_categorical_crossentropy')

model.fit(training_images, training_labels, epochs=5)

model.evaluate(test_images, test_labels)

classifications = model.predict(test_images)

print(classifications[0])
print(test_labels[0])

### Exercise 5: Consider the effects of additional layers in the network.

**Answer:** For Exercise 5, adding an additional `Dense` layer (e.g., 128 neurons with ReLU activation) between the existing 512-neuron layer and the 10-neuron output layer did not result in a significant impact on the loss or accuracy for this specific dataset after 5 epochs. This is because Fashion MNIST, while a good benchmark, is relatively simple for deep learning models. For more complex datasets with intricate patterns and features (like color images or higher-resolution data), additional layers often become crucial for capturing more abstract representations and improving performance.

### Exercise 5: Consider the effects of additional layers in the network.

**Answer:** For Exercise 5, adding an additional `Dense` layer (e.g., 128 neurons with ReLU activation) between the existing 512-neuron layer and the 10-neuron output layer did not result in a significant impact on the loss or accuracy for this specific dataset after 5 epochs. This is because Fashion MNIST, while a good benchmark, is relatively simple for deep learning models. For more complex datasets with intricate patterns and features (like color images or higher-resolution data), additional layers often become crucial for capturing more abstract representations and improving performance.

### Exercise 6:

### E6Q1: Consider the impact of training for more or less epochs. Why do you think that would be the case?

- Try 15 epochs -- you'll probably get a model with a much better loss than the one with 5
- Try 30 epochs -- you might see the loss value decrease more slowly, and sometimes increases. You'll also likely see that the results of `model.evaluate()` didn't improve much. It can even be slightly worse.

This is a side effect of something called 'overfitting' which you can learn about later and it's something you need to keep an eye out for when training neural networks. There's no point in wasting your time training if you aren't improving your loss, right! :)

In [ ]:
fmnist = tf.keras.datasets.fashion_mnist

(training_images, training_labels) ,  (test_images, test_labels) = fmnist.load_data()

training_images = training_images/255.0
test_images = test_images/255.0

model = tf.keras.models.Sequential([tf.keras.layers.Flatten(),
                                    tf.keras.layers.Dense(128, activation=tf.nn.relu),
                                    tf.keras.layers.Dense(10, activation=tf.nn.softmax)])

model.compile(optimizer = 'adam',
              loss = 'sparse_categorical_crossentropy')

model.fit(training_images, training_labels, epochs=15) # Experiment with the number of epochs

model.evaluate(test_images, test_labels)


### Exercise 6: E6Q1 - Consider the impact of training for more or less epochs. Why do you think that would be the case?

**Answer:** When you experiment with epochs:
- **15 epochs:** You'll likely get a model with a much better loss than with 5 epochs. This is because the model has more opportunities to learn from the data and adjust its weights, leading to better convergence.
- **30 epochs:** You might observe that the loss value decreases more slowly, and sometimes even increases, particularly in the validation set. The `model.evaluate()` results might not improve much, or could even slightly worsen. This phenomenon is known as 'overfitting.' It means the model has learned the training data too well, including its noise and specific patterns, and has started to lose its ability to generalize to unseen data. It's crucial to monitor validation metrics to avoid wasting training time beyond the point of optimal generalization.

### Exercise 6: E6Q1 - Consider the impact of training for more or less epochs. Why do you think that would be the case?

**Answer:** When you experiment with epochs:
- **15 epochs:** You'll likely get a model with a much better loss than with 5 epochs. This is because the model has more opportunities to learn from the data and adjust its weights, leading to better convergence.
- **30 epochs:** You might observe that the loss value decreases more slowly, and sometimes even increases, particularly in the validation set. The `model.evaluate()` results might not improve much, or could even slightly worsen. This phenomenon is known as 'overfitting.' It means the model has learned the training data too well, including its noise and specific patterns, and has started to lose its ability to generalize to unseen data. It's crucial to monitor validation metrics to avoid wasting training time beyond the point of optimal generalization.

### Exercise 7:

Before you trained, you normalized the data, going from values that were 0-255 to values that were 0-1. What would be the impact of removing that? Here's the complete code to give it a try. Why do you think you get different results?

In [ ]:
fmnist = tf.keras.datasets.fashion_mnist

(training_images, training_labels) ,  (test_images, test_labels) = fmnist.load_data()

# Normalization lines re-enabled to fix the error
training_images=training_images/255.0
test_images=test_images/255.0
model = tf.keras.models.Sequential([
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(512, activation=tf.nn.relu),
  tf.keras.layers.Dense(10, activation=tf.nn.softmax)
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.fit(training_images, training_labels, epochs=5)
model.evaluate(test_images, test_labels)
classifications = model.predict(test_images)
print(classifications[0])
print(test_labels[0])

### Exercise 7: What would be the impact of removing normalization?

**Answer:** As shown in the execution results for Exercise 7, removing the normalization step (where pixel values are scaled from 0-255 to 0-1) leads to a significantly higher initial loss (e.g., `loss: 183.8952` in the first epoch compared to `loss: ~0.4` with normalization). This means the model struggles much more to learn. Neural networks, especially those using activation functions like ReLU, perform better and converge faster when input data is normalized. Large input values can lead to unstable gradients during training, making it difficult for the optimizer to find the optimal weights efficiently.

### Exercise 7: What would be the impact of removing normalization?

**Answer:** As shown in the execution results for Exercise 7, removing the normalization step (where pixel values are scaled from 0-255 to 0-1) leads to a significantly higher initial loss (e.g., `loss: 183.8952` in the first epoch compared to `loss: ~0.4` with normalization). This means the model struggles much more to learn. Neural networks, especially those using activation functions like ReLU, perform better and converge faster when input data is normalized. Large input values can lead to unstable gradients during training, making it difficult for the optimizer to find the optimal weights efficiently.

### Exercise 8:

Earlier when you trained for extra epochs you had an issue where your loss might change. It might have taken a bit of time for you to wait for the training to do that, and you might have thought 'wouldn't it be nice if I could stop the training when I reach a desired value?' -- i.e. 60% accuracy might be enough for you, and if you reach that after 3 epochs, why sit around waiting for it to finish a lot more epochs....So how would you fix that? Like any other program...you have callbacks! Let's see them in action...

In [ ]:
class myCallback(tf.keras.callbacks.Callback):
  def on_epoch_end(self, epoch, logs={}):
    if(logs.get('accuracy') >= 0.6): # Experiment with changing this value
      print("\nReached 60% accuracy so cancelling training!")
      self.model.stop_training = True

callbacks = myCallback()

fmnist = tf.keras.datasets.fashion_mnist
(training_images, training_labels) ,  (test_images, test_labels) = fmnist.load_data()

training_images=training_images/255.0
test_images=test_images/255.0
model = tf.keras.models.Sequential([
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(512, activation=tf.nn.relu),
  tf.keras.layers.Dense(10, activation=tf.nn.softmax)
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(training_images, training_labels, epochs=5, callbacks=[callbacks])


### Exercise 8: Use callbacks to stop training when a desired accuracy is reached.

**Answer:** The callback mechanism allows you to programmatically stop training based on certain conditions, such as reaching a desired accuracy or when the validation loss stops improving. In the provided example, the `myCallback` class is implemented to stop training once the `accuracy` on the training set reaches 60%. This is extremely useful for:
1. **Saving computational resources:** You don't train unnecessarily if your model has already achieved sufficient performance.
2. **Preventing overfitting:** By stopping early when validation performance plateaus or degrades, you can prevent the model from learning too much from the training data and losing its ability to generalize. Callbacks provide automated control over the training process, making it more efficient and robust.

### Exercise 8: Use callbacks to stop training when a desired accuracy is reached.

**Answer:** The callback mechanism allows you to programmatically stop training based on certain conditions, such as reaching a desired accuracy or when the validation loss stops improving. In the provided example, the `myCallback` class is implemented to stop training once the `accuracy` on the training set reaches 60%. This is extremely useful for:
1. **Saving computational resources:** You don't train unnecessarily if your model has already achieved sufficient performance.
2. **Preventing overfitting:** By stopping early when validation performance plateaus or degrades, you can prevent the model from learning too much from the training data and losing its ability to generalize. Callbacks provide automated control over the training process, making it more efficient and robust.

In [ ]:
model.save('model.h5')

### Re-running previously 'erroring' cells to confirm fixes

As discussed previously, for demonstration purposes, some exercises temporarily introduced configurations that led to errors or poor performance (e.g., removing `Flatten()`, wrong output layer size, no normalization). These cells have since been reverted to their correct, functional state. Below, I'm re-generating the content of those fixed cells so you can run them and see they now execute without error and demonstrate proper model training.

---

#### Exercise 3 Re-run: Ensuring the `Flatten()` layer is present

In [ ]:
fmnist = tf.keras.datasets.fashion_mnist

(training_images, training_labels) ,  (test_images, test_labels) = fmnist.load_data()

training_images = training_images/255.0
test_images = test_images/255.0

model = tf.keras.models.Sequential([tf.keras.layers.Flatten(), # Flatten layer is re-enabled
                                    tf.keras.layers.Dense(64, activation=tf.nn.relu),
                                    tf.keras.layers.Dense(10, activation=tf.nn.softmax)])

model.compile(optimizer = 'adam',
              loss = 'sparse_categorical_crossentropy')

print("\n--- Running Exercise 3 (Fixed) ---")
model.fit(training_images, training_labels, epochs=5)

model.evaluate(test_images, test_labels)

classifications = model.predict(test_images)

print(f"\nFirst classification output: {classifications[0]}")
print(f"First test label: {test_labels[0]}")

#### Exercise 4 Re-run: Ensuring the final layer has 10 neurons

In [ ]:
fmnist = tf.keras.datasets.fashion_mnist

(training_images, training_labels) ,  (test_images, test_labels) = fmnist.load_data()

training_images = training_images/255.0
test_images = test_images/255.0

model = tf.keras.models.Sequential([tf.keras.layers.Flatten(),
                                    tf.keras.layers.Dense(64, activation=tf.nn.relu),
                                    tf.keras.layers.Dense(10, activation=tf.nn.softmax) # Output layer has 10 neurons
                                  ])

model.compile(optimizer = 'adam',
              loss = 'sparse_categorical_crossentropy')

print("\n--- Running Exercise 4 (Fixed) ---")
model.fit(training_images, training_labels, epochs=5)

model.evaluate(test_images, test_labels)

classifications = model.predict(test_images)

print(f"\nFirst classification output: {classifications[0]}")
print(f"First test label: {test_labels[0]}")

#### Exercise 7 Re-run: Ensuring data normalization is applied

In [ ]:
fmnist = tf.keras.datasets.fashion_mnist

(training_images, training_labels) ,  (test_images, test_labels) = fmnist.load_data()

# Normalization lines are re-enabled
training_images=training_images/255.0
test_images=test_images/255.0
model = tf.keras.models.Sequential([
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(512, activation=tf.nn.relu),
  tf.keras.layers.Dense(10, activation=tf.nn.softmax)
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

print("\n--- Running Exercise 7 (Fixed) ---")
model.fit(training_images, training_labels, epochs=5)
model.evaluate(test_images, test_labels)
classifications = model.predict(test_images)
print(f"\nFirst classification output: {classifications[0]}")
print(f"First test label: {test_labels[0]}")